# Cross Encoder testing
Test the performance of cross encoder in pairing soilvoc keywords with record metadata (title, abstract, pdf, ...)

In [ ]:
import json
import torch
from sentence_transformers import CrossEncoder

DOC = """Relative Contribution Of Trees And Crops To Soil Carbon Content In A Parkland System In Burkina Faso Using Variations In Natural C-13 Abundance","The Origin Of Organic Matter Was Studied In The Soils Of A Parkland Of Karite (Vitallaria Paradoxa C.F. Gaertn) And Nere (Parkia Biglobosa (Jacq.) Benth.), Which Is Extensively Cultivated Without The Use Of Fertilisers. In Such Systems, Fertility (Physical, Chemical And Biological) Gradients Around Trees Have Been Attributed By Some Authors To A Priori Differences In Fertility, Allowing For Better Tree Establishment On Richer Sites. In Reverse, Other Workers Believed That These Gradients Are Due To The Contribution Of Trees To The Formation Of Soil Organic Matter Through Litter And Decay Of Roots. Measurements Of The Variations In The C-13 Isotopic Composition Allowed For A Distinction Between Tree (C-3) Derived C And Crop And Grass (C-4) Derived C In The Total Soil Organic C Content. The Organic Carbon Contents Of The Soils Were Recorded Under The Two Species At Two Soil Depths And At Five Distances Going From Tree Trunk To The Open Area And Their C Isotopic Signatures Were Analysed. The Results Showed That Soil Carbon Contents Under Karite (6.43 +/- 0.45 G Kg(-1)) And Nere (5.65 +/- 0.27 G Kg(-1)) Were Significantly Higher (P < 0.01) Than In The Open Area (4.09 +/- 0.26 G Kg(-1)). The Delta C-13 Of Soil C Was Significantly Higher (P < 0.001) In The Open Area (-17.5 +/- 0.3 Parts Per Thousand) Compared With The Values Obtained On Average With Depth And Distance From Tree Under Karite (-20.2 +/- 0.4 Parts Per Thousand) And Nere (-20.1 +/- 0.4 Parts Per Thousand). The C-4-Derived Soil C Was Approximately Constant, And The Differences In Total Soil C Were Fully Explained By The C-3 (Tree) Contributions To Soil Carbon Of 4.01 +/- 0.71, 3.02 +/- 0.53, 1.53 +/- 0.10 G Kg(-1), Respectively Under Karite, Nere And In The Open Area. These Results Show That Trees In Parklands Have A Directly Positive Contribution To Soil Carbon Content, Justifying The Need To Encourage The Maintenance Of Trees In These Systems In Semi-Arid Environments Where The Carbon Content Of Soil Appears To Be The First Limiting Factor For Crop Growth.
"""

with open("../concepts_multilingual.json", encoding="utf-8") as f:
    concepts = json.load(f)

# One (label, concept) entry per English label; several labels share a concept.
labels = [(lab, c["identifier"].split("#")[-1])
          for c in concepts for lab in c["labels"].get("en", [])]
print(f"{len(labels)} English labels from {len(concepts)} concepts")

model = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
                     activation_fn=torch.nn.Sigmoid(), max_length=256)
scores = model.predict([(lab, DOC) for lab, _ in labels], show_progress_bar=True)
for score, (lab, cid) in sorted(zip(scores, labels), reverse=True)[:10]:
    print(f"  {score:.4f}  {lab:<32} {cid}")



1064 English labels from 799 concepts


Batches: 100%|██████████| 34/34 [01:23<00:00,  2.46s/it]

  0.8791  soil organic matter content      SoilOrganicMatterContents
  0.8339  soil organic matter contents     SoilOrganicMatterContents
  0.7613  soil organic matter              SoilOrganicMatter
  0.7025  soil organic carbon              SoilOrganicCarbon
  0.4383  soil organic components          SoilOrganicComponents
  0.4254  soil organic component           SoilOrganicComponents
  0.2185  critical soil organic matter content CriticalSoilOrganicMatterContents
  0.1831  soil inorganic carbon            SoilInorganicCarbon
  0.1815  soil organic matter class        SoilOrganicMatterClass
  0.1516  critical soil organic matter contents CriticalSoilOrganicMatterContents


In [6]:
# Now try to do with the german content.
DOC_DE = """
Die Gesamt-Phosphoreinträge in die Gewässer wurden mit dem Stoffflussmodell MODIFFUS über alle diffusen Eintragsquellen (Ackerland, Dauergrünland, Wald, Gletscher, Siedlungsgrünflächen etc.) und alle diffusen Eintragspfade (Bodenerosion, Auswaschung, Abschwemmung, Drainage, atmosphärische Deposition etc.) berechnet. Die Karte zeigt die aufsummierten Verluste pro Landnutzungskategorie im Hektarraster, basierend auf der Arealstatistik 2013/18. Es wurden mittlere klimatische Bedingungen zugrunde gelegt, das Bezugsjahr ist 2020.
"""
scores = model.predict([(lab, DOC_DE) for lab, _ in labels], show_progress_bar=True)

for score, (lab, cid) in sorted(zip(scores, labels), reverse=True)[:10]:
    print(f"  {score:.4f}  {lab:<32} {cid}")

Batches: 100%|██████████| 34/34 [00:48<00:00,  1.43s/it]

  0.8466  phosphorus total elements        PhosphorusTotalElements
  0.6913  soil phosphorus loss             SoilPLoss
  0.5186  soil water loss                  SoilWaterLoss
  0.4520  soil P loss                      SoilPLoss
  0.4472  soil water deficit               SoilMoistureDeficit
  0.4143  soil particle movement           SoilParticleMovement
  0.4046  land use class                   LandUseClass
  0.3834  soil organic carbon loss         SOCLoss
  0.3286  soil water contents              SoilWaterContents
  0.3022  soil oxygen contents             SoilOxygenContents


CE doing better than KeyBERT in pairing content with keywords in different languages (de - en). The result looks ok, but Phosphorus (P) is not there.